# Homework 04 - Module 04 evaluation

> Note: sometimes your answer doesn’t match one of the options exactly.
> That’s fine. Select the option that’s closest to your solution.

In this homework, we will use the lead scoring dataset Bank Marketing
dataset. Download it from
[here](https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv).

In this dataset our desired target for classification task will be
`converted` variable - has the client signed up to the platform or not.

# import packages

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
pd.set_option('max_colwidth', 1000)
pd.set_option("display.precision", 4)

# load data

In [3]:
path="https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv"

In [4]:
dataset = pd.read_csv(path)

## Data preparation

Check if the missing values are presented in the features.
-   If there are missing values:
    -   For caterogiral features, replace them with ‘NA’
    -   For numerical features, replace with with 0.0

Split the data into 3 parts: train/validation/test with 60%/20%/20%
distribution. Use `train_test_split` function for that with
`random_state=1`

In [6]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1462 entries, 0 to 1461
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               1334 non-null   object 
 1   industry                  1328 non-null   object 
 2   number_of_courses_viewed  1462 non-null   int64  
 3   annual_income             1281 non-null   float64
 4   employment_status         1362 non-null   object 
 5   location                  1399 non-null   object 
 6   interaction_count         1462 non-null   int64  
 7   lead_score                1462 non-null   float64
 8   converted                 1462 non-null   int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 102.9+ KB


In [5]:
dataset.head(5)

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NaN,1,79450.0,unemployed,south_america,4,0.93999999999999994671,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80000000000000004441,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.68999999999999994671,1
3,paid_ads,retail,2,83843.0,NaN,australia,1,0.86999999999999999556,0
4,referral,education,3,85012.0,self_employed,europe,3,0.61999999999999999556,1


In [7]:
dataset.isnull().sum()

lead_source                 128
industry                    134
number_of_courses_viewed      0
annual_income               181
employment_status           100
location                     63
interaction_count             0
lead_score                    0
converted                     0
dtype: int64

In [8]:
df=dataset.copy()

In [116]:
categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)

numerical_columns=list(df.dtypes[df.dtypes=='float64'].index)

for a in list(df.dtypes[df.dtypes=='int64'].index):
    numerical_columns.append(a)
    

print("categorical_columns:",categorical_columns)
print("numerical_columns:",numerical_columns)

categorical_columns: ['lead_source', 'industry', 'employment_status', 'location']
numerical_columns: ['annual_income', 'lead_score', 'number_of_courses_viewed', 'interaction_count', 'converted']


In [33]:
for var in df.columns:
    
    if df[var].dtypes== 'object':
        df[var]=df[var].fillna(value='NA')   

    else:
        df[var]=df[var].fillna(value=0)
            

lead_source
industry
number_of_courses_viewed
annual_income
employment_status
location
interaction_count
lead_score
converted


"\nfor var in categorical_columns:\n\n    if df[var].isnull().sum()>0:\n        df[var]=df[var].fillna(value='NA')\n"

In [75]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

In [78]:
print('total',df.shape)
print('full train',df_full_train.shape)
print('train',df_train.shape)
print('val',df_val.shape)
print('test',df_test.shape)

print('---')
print('total:',df.shape[0])
print('train',df.shape[0]*0.6)
print('val',df.shape[0]*0.2)
print('test',df.shape[0]*0.2)


total (1462, 9)
full train (1169, 9)
train (876, 9)
val (293, 9)
test (293, 9)
---
total: 1462
train 877.1999999999999
val 292.40000000000003
test 292.40000000000003


In [79]:
df_full_train = df_full_train.reset_index(drop=True)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [81]:
y_full_train = df_full_train.converted.values
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_full_train['converted']
del df_train['converted']
del df_val['converted']
del df_test['converted'] 

In [ ]:
numerical_columns.remove('converted')

In [150]:
from sklearn.feature_extraction import DictVectorizer
 
train_dicts = df_train[categorical_columns + numerical_columns].to_dict(orient='records')
dv = DictVectorizer(sparse=False)
 
dv.fit(train_dicts)
X_train = dv.transform(train_dicts)
# instead of last two lines, you can also use
# X_train = dv.fit_transform(train_dicts)
 
X_train.shape

(876, 31)

In [151]:
val_dicts = df_val[categorical_columns + numerical_columns].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [152]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'liblinear'
,max_iter,1000
,multi_class,'deprecated'


In [132]:
#model.coef_[0].round(5)
#model.intercept_
#model.predict(X_train)
#model.predict_proba(X_train)

In [162]:
#applying to validation set
y_pred = model.predict_proba(X_val)[:,1]
y_pred
print("")

In [163]:
converted_decision = y_pred >0.5

In [171]:
accuracy_general=(y_val == converted_decision.astype(int)).mean()

accuracy_general

np.float64(0.6996587030716723)

## Questions

### Question 1: ROC AUC feature importance

ROC AUC could also be used to evaluate feature importance of numerical
variables.

Let’s do that

-   For each numerical variable, use it as score (aka prediction) and
    compute the AUC with the `y` variable as ground truth.
-   Use the training dataset for that

If your AUC is \< 0.5, invert this variable by putting “-” in front

(e.g. `-df_train['balance']`)

AUC can go below 0.5 if the variable is negatively correlated with the
target variable. You can change the direction of the correlation by
negating this variable - then negative correlation becomes positive.

Which numerical variable (among the following 4) has the highest AUC?

-   `lead_score`
-   `number_of_courses_viewed`
-   `interaction_count`
-   `annual_income`

### Question 2: Training the model

Apply one-hot-encoding using `DictVectorizer` and train the logistic
regression with these parameters:

``` python
LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
```

What’s the AUC of this model on the validation dataset? (round to 3
digits)

-   0.32
-   0.52
-   0.72
-   0.92

### Question 3: Precision and Recall

Now let’s compute precision and recall for our model.

-   Evaluate the model on all thresholds from 0.0 to 1.0 with step 0.01
-   For each threshold, compute precision and recall
-   Plot them

At which threshold precision and recall curves intersect?

-   0.145
-   0.345
-   0.545
-   0.745

### Question 4: F1 score

Precision and recall are conflicting - when one grows, the other goes
down. That’s why they are often combined into the F1 score - a metrics
that takes into account both

This is the formula for computing F1:

$$F_1 = 2 \cdot \cfrac{P \cdot R}{P + R}$$

Where $P$ is precision and $R$ is recall.

Let’s compute F1 for all thresholds from 0.0 to 1.0 with increment 0.01

At which threshold F1 is maximal?

-   0.14
-   0.34
-   0.54
-   0.74

### Question 5: 5-Fold CV

Use the `KFold` class from Scikit-Learn to evaluate our model on 5
different folds:

    KFold(n_splits=5, shuffle=True, random_state=1)

-   Iterate over different folds of `df_full_train`
-   Split the data into train and validation
-   Train the model on train with these parameters:
    `LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)`
-   Use AUC to evaluate the model on validation

How large is standard deviation of the scores across different folds?

-   0.0001
-   0.006
-   0.06
-   0.36

### Question 6: Hyperparameter Tuning

Now let’s use 5-Fold cross-validation to find the best parameter `C`

-   Iterate over the following `C` values: `[0.000001, 0.001, 1]`
-   Initialize `KFold` with the same parameters as previously
-   Use these parameters for the model:
    `LogisticRegression(solver='liblinear', C=C, max_iter=1000)`
-   Compute the mean score as well as the std (round the mean and std to
    3 decimal digits)

Which `C` leads to the best mean score?

-   0.000001
-   0.001
-   1

If you have ties, select the score with the lowest std. If you still
have ties, select the smallest `C`.

## Submit the results

-   Submit your results here:
    https://courses.datatalks.club/ml-zoomcamp-2025/homework/hw04
-   If your answer doesn’t match options exactly, select the closest one